---

## Pre-Submission Checklist

- [ ] Name and student ID at top
- [ ] No cells are added or removed
- [ ] All TODO sections completed
- [ ] All questions answered
- [ ] Code runs without errors
- [ ] Results tables included
- [ ] Run All before saving

## Setup and Imports

In [1]:
# Standard libraries
import numpy as np
import random
from collections import defaultdict

# NLTK for corpus and HMM
import nltk
from nltk.tag import hmm

# Scikit-learn for cross-validation
from sklearn.model_selection import KFold

# Set random seed for reproducibility
seed = 42
np.random.seed(seed)
random.seed(seed)

---

# Task 1: HMM-based POS Tagging 

In this part, you will implement three different approaches to training HMM models for POS tagging:
1. **Supervised learning**: Uses fully labelled data
2. **Unsupervised learning**: Uses unlabelled data with Baum-Welch algorithm
3. **Semi-supervised learning**: Combines a small amount of labelled data with larger unlabelled data

## 1.1: Load the Brown Corpus

Load the Brown corpus with universal POS tagset. Start with 5000 sentences for testing, then increase to the maximum your environment can handle.

In [2]:
# Download required NLTK data
from nltk.corpus import brown
nltk.download('brown')
nltk.download('universal_tagset')

NUM_SENTENCES = 5000  # Start with 5000 sentences and increase later

all_data = list(brown.tagged_sents(tagset="universal"))[:5000]

print(f"Total sentences loaded: {len(all_data)}")
print(f"\nExample sentence:")
print(all_data[0])

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


Total sentences loaded: 5000

Example sentence:
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


## 1.2: Remove Tags (2 points)

Implement the `remove_tags()` function that converts tagged sentences to untagged format required by NLTK's unsupervised training.

**Input:** List of tagged sentences `[[(word, tag), ...], ...]`  
**Output:** List of untagged sentences `[[(word, None), ...], ...]`

In [3]:
def remove_tags(tagged_sents):

    """
    Remove tags from tagged sentences to create untagged data.


    Args:
        tagged_sents: List of tagged sentences [[(word, tag), ...], ...]

    Returns:
        List of untagged sentences [[(word, None), ...], ...] (required format for NLTK)
    """
    # TODO: Implement this function
    untagged = []
    for sent in tagged_sents:
        new_sent = []
        for (w, t) in sent:
            new_sent.append((w, None))
        untagged.append(new_sent)
    return untagged



    pass

## 1.3: Evaluate Tagger (5 points)

Implement the `evaluate_tagger()` function that evaluates a trained tagger on test data and returns per-tag accuracy statistics.

**Input:**
- `tagger`: Trained HMM tagger
- `test_data`: List of tagged test sentences

**Output:** Dictionary with per-tag statistics `{tag: {'correct': count, 'total': count}}`

In [4]:
def evaluate_tagger(tagger, test_data):
    """
    Evaluate tagger and return per-tag accuracy.

    Args:
        tagger: Trained HMM tagger
        test_data: List of tagged sentences for testing

    Returns:
        dict of {tag: {'correct': count, 'total': count}}
    """
    # TODO: Implement this function
    #
    # Steps:
    # 1. Initialize tag_stats as defaultdict(lambda: {'correct': 0, 'total': 0})
    # 2. For each sentence in test_data:
    #    a. Extract words and true tags
    #    b. Use tagger.tag(words) to get predicted tags
    #    c. Compare true vs predicted tags
    #    d. Update tag_stats accordingly
    # 3. Handle exceptions (if tagging fails, count all as incorrect)
    # 4. Return tag_stats
    correct=0
    incorrect=0
    total=0
    tag_stats = defaultdict(lambda: {'correct': 0, 'total': 0})
    for ele in test_data:

        words = [word for (word, tag) in ele]
        tags = [tag for (word, tag) in ele]


        try:
            predicted = tagger.tag(words)
            ptags = [p[1] for p in predicted]
        except Exception:

            ptags = [None] * len(tags)


        i = 0
        while i < len(tags):
            true_t = tags[i]
            pred_t = ptags[i]

            tag_stats[true_t]['total'] += 1

            if true_t == pred_t:
                tag_stats[true_t]['correct'] += 1
                correct += 1
            else:
                incorrect += 1

            total += 1
            i += 1

    return tag_stats


    pass

## 1.4: Semi-supervised HMM Training (20 points)

Implement semi-supervised HMM training that combines a small amount of labelled data (1%) with larger unlabelled data (99%).

**Steps:**
1. Split data: 1% tagged, 99% untagged
2. Train supervised model on 1% tagged data
3. Use this model to initialise Baum-Welch on 99% untagged data
4. Refine with unsupervised learning

In [10]:
def train_semi_supervised(tagged_data, percent_tagged=1.0):

    """
    Train HMM with a mix of tagged and untagged data.

    Args:
        tagged_data: List of tagged sentences
        percent_tagged: Percentage of data to keep tags (default 1.0%)

    Returns:
        Trained HMM tagger
    """
    # TODO: Implement semi-supervised training
    #
    # Steps:
    # 1. Calculate split index: int(len(tagged_data) * (percent_tagged / 100.0))
    #    Ensure at least 1 sentence is tagged
    #
    # 2. Split data:
    #    tagged_portion = tagged_data[:split_idx]
    #    untagged_portion = remove_tags(tagged_data[split_idx:])
    #
    # 3. Extract states and symbols from ALL data
    #    NOTE: In fact, we should have used only the tagged_data for this,
    #    but we are not sure if that portion includes all the tags and symbols
    #
    # 4. Initialize trainer with states and symbols using
    #    hmm.HiddenMarkovModelTrainer
    #
    # 5. Train supervised model on small tagged data using trainer.train_supervised
    #
    # 6. Refine with Baum-Welch on untagged data using trainer.train_unsupervised
    #
    # 7. Return refined_tagger (or error if no untagged data or no tagged data)
    check = 0

    split_idx = int(len(tagged_data) * (percent_tagged / 100.0))

    if split_idx >= 1:
        check = 1
    else:
        split_idx=1
    tagged_portion = tagged_data[:split_idx]
    untagged_portion = remove_tags(tagged_data[split_idx:])
    states = [ele[1] for sub in tagged_data for ele in sub]
    symbols = [ele[0] for sub in tagged_data for ele in sub]
    setstates=set(states)
    trainer = hmm.HiddenMarkovModelTrainer(setstates, symbols)
    model1 = trainer.train_supervised(tagged_portion)
    refined_tagger = trainer.train_unsupervised(untagged_portion, model=model1,max_iterations=10)
    if len(tagged_portion) == 0 or len(untagged_portion) == 0:
        raise Exception("Sorry, no tagged_portion or untagged_portion ")
    return  refined_tagger



    pass

## 1.5: 5-Fold Cross-Validation (25 Points)

Implement 5-fold cross-validation to train and evaluate all three models. This ensures robust performance estimates.

**Steps:**
1. Split data into 5 folds
2. For each fold:
   - Train
    - supervised (`trainer.train_supervised()`)
    - unsupervised (`trainer.train_unsupervised()`)
    - semi-supervised models (`train_semi_supervised()`)
   - Evaluate on test fold
   - Accumulate results
3. Calculate average accuracy across all folds

In [11]:
# Prepare for 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=seed)

# Storage for results across folds
results = {
    'unsupervised': defaultdict(lambda: {'correct': 0, 'total': 0}),
    'supervised': defaultdict(lambda: {'correct': 0, 'total': 0}),
    'semi_supervised': defaultdict(lambda: {'correct': 0, 'total': 0})
}

# TODO: Complete the cross-validation loop
#
# for train_idx, test_idx in kf.split(all_data):
#     print(f"{'='*60}")
#     print(f"FOLD {fold_num}")
#     print(f"{'='*60}")
#
#     # Split data to train_data and test_data
#     ...
#
#     print(f"Training set: {len(train_data)} sentences")
#     print(f"Test set: {len(test_data)} sentences")
#
#     # Extract all_tags and all_symbols from training data
#      ...
#
#
#     # 1. Unsupervised HMM (Baum-Welch)
#     print("\n1. Training Unsupervised HMM (Baum-Welch)...")
#     try:
#         1.1 Convert to untagged format
#
#         1.2 Initialize trainer with states and symbols
#
#         1.3 Train unsupervised
#
#         1.4 Evaluate unsupervised tagger and update the results dict
#
#         print("Unsupervised training complete")
#     except Exception as e:
#         print(f"ERROR: Unsupervised training failed: {e}")
#         import traceback
#         traceback.print_exc()
#
#     # 2. Supervised HMM
#     print("\n2. Training Supervised HMM...")
#     try:
#         1.1 Initialize trainer with states and symbols
#
#         1.2 Train supervised
#
#         1.3 Evaluate supervised tagger and update the results dict
#
#         print("Supervised training complete")
#     except Exception as e:
#         print(f"ERROR: Supervised training failed: {e}")
#         import traceback
#         traceback.print_exc()
#
#     # 3. Semi-supervised HMM (1% tagged, 99% untagged)
#     print("\n3. Training Semi-Supervised HMM (1% tagged, 99% untagged)...")
#     try:
#         1.1 Train semi-supervised
#
#         1.3 Evaluate semi-supervised tagger and update the results dict
#
#         print("Semi-supervised training complete")
#     except Exception as e:
#         print(f"ERROR: Semi-supervised training failed: {e}")
#         import traceback
#         traceback.print_exc()


fold_num = 1

for train_idx, test_idx in kf.split(all_data):
    print(f"{'='*60}")
    print(f"FOLD {fold_num}")
    print(f"{'='*60}")

    train = [all_data[i] for i in train_idx]
    test  = [all_data[i] for i in test_idx]

    print(f"Training set: {len(train)} sentences")
    print(f"Test set: {len(test)} sentences")

    print("\n1. Training Unsupervised HMM (Baum-Welch)...")


    all_symbols = []
    all_tags = []
    for sentence in train:
        for (word, tag) in sentence:
            all_symbols.append(word)
            all_tags.append(tag)

    all_symbols = list(set(all_symbols))
    all_tags    = list(set(all_tags))


    try:
        unlabeled_sentences = remove_tags(train)

        trainer_unsup = hmm.HiddenMarkovModelTrainer(all_tags, all_symbols)
        tagger_unsup = trainer_unsup.train_unsupervised(unlabeled_sentences,max_iterations=10)
        stats_unsup = evaluate_tagger(tagger_unsup, test)

        for tag, counts in stats_unsup.items():
            results['unsupervised'][tag]['correct'] += counts['correct']
            results['unsupervised'][tag]['total']   += counts['total']

    except Exception as e:
        print(f"ERROR: Unsupervised training failed: {e}")
        import traceback
        traceback.print_exc()


    try:
        print("\n2. Training Supervised HMM...")

        trainer_sup = hmm.HiddenMarkovModelTrainer(all_tags, all_symbols)
        tagger_sup = trainer_sup.train_supervised(train)

        stats_sup = evaluate_tagger(tagger_sup, test)

        for tag, counts in stats_sup.items():
            results['supervised'][tag]['correct'] += counts['correct']
            results['supervised'][tag]['total']   += counts['total']

    except Exception as e:
        print(f"ERROR: supervised training failed: {e}")
        import traceback
        traceback.print_exc()


    try:
        print("\n3. Training Semi-Supervised HMM...")

        model_semi = train_semi_supervised(train, percent_tagged=1.0)
        stats_semi = evaluate_tagger(model_semi, test)

        for tag, counts in stats_semi.items():
            results['semi_supervised'][tag]['correct'] += counts['correct']
            results['semi_supervised'][tag]['total']   += counts['total']

    except Exception as e:
        print(f"ERROR: train semi supervised training failed: {e}")
        import traceback
        traceback.print_exc()

    fold_num = fold_num + 1



















FOLD 1
Training set: 4000 sentences
Test set: 1000 sentences

1. Training Unsupervised HMM (Baum-Welch)...
iteration 0 logprob -1192724.0350421006
iteration 1 logprob -878683.1782741487
iteration 2 logprob -877549.965593567
iteration 3 logprob -876397.7886241373
iteration 4 logprob -875019.8902629213
iteration 5 logprob -873240.1083867733
iteration 6 logprob -870865.1815736893
iteration 7 logprob -867659.5524945226
iteration 8 logprob -863352.7581165859
iteration 9 logprob -857733.8555412923

2. Training Supervised HMM...

3. Training Semi-Supervised HMM...
iteration 0 logprob -4.484600000000029e+304
iteration 1 logprob -6.391436595342868e+289
iteration 2 logprob -892037.1966913621
iteration 3 logprob -865828.7819085392
iteration 4 logprob -863749.4419701553
iteration 5 logprob -861661.0586968748
iteration 6 logprob -859276.9500922204
iteration 7 logprob -856462.0906269127
iteration 8 logprob -853174.3098113051
iteration 9 logprob -849382.28524689
FOLD 2
Training set: 4000 sentences
Te

## 1.6: Display Results 

Display the average across folds results in a formatted table showing
- tag frequency (%)
- per-tag accuracy (%)
- overall accuracy (%)


1.   List item
2.   List item


for all three models.

In [12]:
import pandas as pd

totla = []

for modeltype, modelcount in results.items():
    total_tokens = sum(modelcount[tag]['total'] for tag in modelcount)
    total_correct = sum(modelcount[tag]['correct'] for tag in modelcount)
    overall_accuracy = (total_correct / total_tokens) * 100 if total_tokens > 0 else 0

    for tag, counts in modelcount.items():
        tag_total = counts['total']
        tag_correct = counts['correct']

        tag_frequency = (tag_total / total_tokens) * 100 if total_tokens > 0 else 0
        per_tag_accuracy = (tag_correct / tag_total) * 100 if tag_total > 0 else 0

        totla.append({
            'Model': modeltype,
            'Tag': tag,
            'Tag Frequency (%)': tag_frequency,
            'Per-Tag Accuracy (%)': per_tag_accuracy,
            'Overall Accuracy (%)': overall_accuracy
        })

df = pd.DataFrame(totla)
df = df.sort_values(by=['Model', 'Tag'])
print(df)


              Model   Tag  Tag Frequency (%)  Per-Tag Accuracy (%)  \
25  semi_supervised     .          11.682041              5.140923   
32  semi_supervised   ADJ           6.821422             13.145477   
34  semi_supervised   ADP          12.318474             81.125877   
24  semi_supervised   ADV           3.422207              8.949207   
33  semi_supervised  CONJ           2.727833              7.956844   
26  semi_supervised   DET          11.420846             12.506040   
27  semi_supervised  NOUN          30.172628              7.013747   
30  semi_supervised   NUM           2.061050             12.226685   
29  semi_supervised  PRON           2.560447             35.560345   
31  semi_supervised   PRT           2.252348             16.496529   
28  semi_supervised  VERB          14.469654             24.941206   
35  semi_supervised     X           0.091050              0.000000   
13       supervised     .          11.682041             33.451425   
20       supervised 

## 1.7: Sample Predictions

Test the models on sample sentences to see how it performs.

In [13]:
# Print sample predictions from the last fold
print("Sample predictions from last fold:")
sample_sentences = [
    "Today is a good day .",
    "Joe met Joanne in Delhi .",
    "Time flies like an arrow .",
    "The good , the bad , the ugly went to a bar ."
]

print(f"{'='*10} Spervised HMM Tagger {'='*10}")
for sent in sample_sentences:
    try:
        tagged = tagger_sup.tag(sent.split())
        print(f"  {sent}")
        print(f"  → {tagged}\n")
    except Exception as e:
        print(f"  {sent}")
        print(f"ERROR: Tagging failed: {e}\n")

print(f"{'='*10} Unspervised HMM Tagger {'='*10}")
for sent in sample_sentences:
    try:
        tagged = tagger_unsup.tag(sent.split())
        print(f"  {sent}")
        print(f"  → {tagged}\n")
    except Exception as e:
        print(f"  {sent}")
        print(f"ERROR: Tagging failed: {e}\n")

print(f"{'='*10} Semi-spervised HMM Tagger {'='*10}")
for sent in sample_sentences:
    try:
        tagged = model_semi.tag(sent.split())
        print(f"  {sent}")
        print(f"  → {tagged}\n")
    except Exception as e:
        print(f"  {sent}")
        print(f"ERROR: Tagging failed: {e}\n")

Sample predictions from last fold:
========== Spervised HMM Tagger ==========
  Today is a good day .
  → [('Today', 'NOUN'), ('is', 'VERB'), ('a', 'DET'), ('good', 'ADJ'), ('day', 'NOUN'), ('.', '.')]

  Joe met Joanne in Delhi .
  → [('Joe', 'NOUN'), ('met', 'VERB'), ('Joanne', 'NOUN'), ('in', 'ADP'), ('Delhi', 'NOUN'), ('.', '.')]

  Time flies like an arrow .
  → [('Time', 'NOUN'), ('flies', 'VERB'), ('like', 'ADP'), ('an', 'DET'), ('arrow', 'ADP'), ('.', 'ADP')]

  The good , the bad , the ugly went to a bar .
  → [('The', 'DET'), ('good', 'ADJ'), (',', '.'), ('the', 'DET'), ('bad', 'ADJ'), (',', '.'), ('the', 'DET'), ('ugly', 'ADJ'), ('went', 'VERB'), ('to', 'ADP'), ('a', 'DET'), ('bar', 'NOUN'), ('.', '.')]

========== Unspervised HMM Tagger ==========
  Today is a good day .
  → [('Today', 'ADJ'), ('is', 'NOUN'), ('a', 'CONJ'), ('good', 'VERB'), ('day', 'CONJ'), ('.', 'VERB')]

  Joe met Joanne in Delhi .
  → [('Joe', 'VERB'), ('met', 'NUM'), ('Joanne', 'ADJ'), ('in', 'DET'), (

/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


---

# Task 2: Text Generation with HMMs 

In this part, you will use a trained HMM to generate text. The idea is to sample from the learned transition and emission probabilities to create new sequences.

## 2.1: Train HMM Model for Generation

First, train a supervised HMM model on the Brown corpus for text generation. We'll normalise words to lowercase for better generation.

In [14]:
def train_hmm_model(num_sentences=5000):
    """Train an HMM model on Brown corpus for text generation"""
    print("Loading Brown corpus...")
    tagged_sents = list(brown.tagged_sents(tagset="universal"))[:num_sentences]

    print(f"Training HMM on {len(tagged_sents)} sentences...")

    # Extract states and symbols
    all_tags = set()
    all_symbols = set()
    for sent in tagged_sents:
        for word, tag in sent:
            all_tags.add(tag)
            all_symbols.add(word.lower())  # Normalise to lowercase

    # Normalise training data to lowercase
    normalized_sents = []
    for sent in tagged_sents:
        normalized_sents.append([(word.lower(), tag) for word, tag in sent])

    # Train the model
    trainer = hmm.HiddenMarkovModelTrainer(
        states=list(all_tags),
        symbols=list(all_symbols)
    )

    tagger = trainer.train_supervised(normalized_sents)

    print("Training complete!")
    return tagger

# Train the model
tagger_gen = train_hmm_model(num_sentences=20000)

# Show model statistics
print(f"\nModel Statistics:")
print(f"  - Number of states (POS tags): {len(tagger_gen._states)}")
print(f"  - Number of symbols (words): {len(tagger_gen._symbols)}")
print(f"  - States: {', '.join(sorted(tagger_gen._states))}")

Loading Brown corpus...
Training HMM on 20000 sentences...
Training complete!

Model Statistics:
  - Number of states (POS tags): 12
  - Number of symbols (words): 30743
  - States: ., ADJ, ADP, ADV, CONJ, DET, NOUN, NUM, PRON, PRT, VERB, X


## 2.2: Implement State Sampling (7 points)

Implement the `sample_state()` function that samples the next POS tag based on transition probabilities.

**Temperature parameter:** Controls randomness
- Lower temperature (e.g., 0.5): More conservative, follows high-probability transitions
- Higher temperature (e.g., 2.0): More creative, explores diverse transitions

In [24]:
def sample_state(tagger, current_state, temperature=1.0):
    """
    Sample next state from transition distribution.

    Args:
        tagger: Trained HMM tagger
        current_state: Current POS tag
        temperature: Controls randomness (default 1.0)

    Returns:
        Next POS tag (state)
    """
    # TODO: Implement state sampling
    #
    # Steps:
    # 1. Get all possible states from tagger._states
    #
    # 2. For each state, get transition probability from current_state
    #    using: tagger._transitions[current_state].logprob(state)
    #
    # 3. Apply temperature scaling: probs = probs / temperature
    #
    # 4. Convert from log2 probabilities to regular probabilities: 2 ** probs
    #
    # 5. Normalise probabilities to sum to 1
    #
    # 6. Sample using np.random.choice(states, p=probs)
    states=tagger._states
    transitions=[]
    for state in states:
     ts= tagger._transitions[current_state].logprob(state)
     transitions.append(ts)
    transitions = np.array(transitions)
    transitions = transitions / temperature
    transitions=2**transitions
    Normalise=sum(transitions)
    i=0
    for i in range(len(transitions)):
        transitions[i] = transitions[i] / Normalise
    return np.random.choice(states, p=transitions)




    pass

## 2.3: Implement Word Sampling 

Implement the `sample_word()` function that samples a word given a POS tag based on emission probabilities.

In [25]:
def sample_word(tagger, state, temperature=1.0):
    """
    Sample word from output distribution of given state.

    Args:
        tagger: Trained HMM tagger
        state: Current POS tag
        temperature: Controls randomness (default 1.0)

    Returns:
        Sampled word
    """
    # TODO: Implement word sampling
    #
    # Steps:
    # 1. Get all possible symbols (words) from tagger._symbols
    #
    # 2. For each symbol, get emission probability from state
    #    using: tagger._outputs[state].logprob(symbol)
    #
    # 3. Apply temperature scaling: probs = probs / temperature
    #
    # 4. Convert from log2 probabilities to regular probabilities: 2 ** probs
    #
    # 5. Handle infinities (words with zero probability) using np.nan_to_num
    #
    # 6. Normalise probabilities to sum to 1
    #    (if all are zero, use uniform distribution)
    #
    # 7. Sample using np.random.choice(symbols, p=probs)
    symbols=tagger._symbols
    transitions=[]
    for symbol in symbols:
     ts = tagger._outputs[state].logprob(symbol)
     transitions.append(ts)
    transitions = np.array(transitions)
    transitions = transitions / temperature
    transitions=2**transitions
    transitions=np.nan_to_num(transitions)
    i=0
    Normalise=sum(transitions)
    Normalise = transitions.sum()
    if Normalise == 0:
          transitions = np.ones(len(transitions)) / len(transitions)
    else:
          transitions = transitions / Normalise

    return np.random.choice(symbols, p=transitions)



    pass

## 2.4: Implement Text Generation 

Implement the `generate_text_from_word()` function that generates text starting from a given word.

In [26]:
def generate_text_from_word(tagger, start_word, length=20, temperature=1.0):
    """
    Generate text starting with a given word using the HMM model.

    Args:
        tagger: Trained HMM tagger
        start_word: Word to start the generation
        length: Number of words to generate
        temperature: Controls randomness (higher = more random)

    Returns:
        List of generated words
    """
    # TODO: Implement text generation
    #
    # Steps:
    # 1. Normalise start_word to lowercase
    #
    # 2. Check if start_word is in vocabulary (tagger._symbols)
    #    - If not, find similar words or use random word
    #
    # 3. Initialise generated list with start_word
    #
    # 4. Infer the most likely tag for start_word:
    #    - For each state, get emission probability
    #    - Choose state with highest probability
    #
    # 5. Generate subsequent words in a loop:
    #    - Sample next state using sample_state()
    #    - Sample word from that state using sample_word()
    #    - Append word to generated list
    #    - Update current_state
    #
    # 6. Return generated list
    start_word=start_word.lower()
    symbols = list(tagger._symbols)
    states=list(tagger._states)

    if start_word  in  tagger._symbols:
      start_word=start_word
    else:
      idx = random.randint(0, len(symbols) - 1)
      start_word =symbols[idx]
    generated = [start_word]
    a=0
    maxprob=-1111111111111111111111111
    maxstatetz=states[0]
    for st in tagger._states:
      a=tagger._output_logprob(st, start_word)
      if(maxprob<a):
        maxprob=a
        maxstatetz=st
    i=0
    current_state=maxstatetz
    for i in range(length - 1):
      next_state= sample_state(tagger, current_state, temperature)
      next_word= sample_word(tagger, next_state, temperature)
      generated.append(next_word)
      current_state = next_state
    return generated




    pass

## 2.6: Test Text Generation

Now test your text generation functions with different starting words and temperatures.

In [27]:
# Test with example words
print("="*70)
print("EXAMPLE GENERATIONS")
print("="*70)

example_words = ["the", "dog", "running", "beautiful", "computer", "yesterday"]

for word in example_words:
    print(f"\n--- Starting with '{word}' ---")
    try:
        words = generate_text_from_word(tagger_gen, word, length=15, temperature=1.0)
        print(" ".join(words))
    except Exception as e:
        print(f"Error: {e}")

EXAMPLE GENERATIONS

--- Starting with 'the' ---
the table help with uninjured renunciations that emperor above the faith his 450 . and

--- Starting with 'dog' ---
dog ? efforts along went `` miami in roman . , any recognized can structure

--- Starting with 'running' ---
running into residential time certainly central innocent that trustees of belief morning `` it include

--- Starting with 'beautiful' ---
beautiful will moiseyev will of slaughter machines , with firmer ballad choose the famous an

--- Starting with 'computer' ---
computer went bob france seem him is tall harvard in a synchronous secrets . ;

--- Starting with 'yesterday' ---
yesterday overturned a nine arts as who , built well out-moded order villa of any


In [28]:
# Test different temperatures
print("\n" + "="*70)
print("TEMPERATURE COMPARISON")
print("="*70)

test_word = "the"
temperatures = [("Conservative", 0.5), ("Balanced", 1.0), ("Creative", 2.0)]

for temp_name, temp_value in temperatures:
    print(f"\n{temp_name} (temp={temp_value}):")
    words = generate_text_from_word(tagger_gen, test_word, length=20, temperature=temp_value)
    print(" ".join(words))


TEMPERATURE COMPARISON

Conservative (temp=0.5):
the other terms , prokofieff , is a mr. of manager , of business , land was when possible something

Balanced (temp=1.0):
the 6-foot-10 while the outlook lonsdale to moves education to a was cannot it backed satisfactory ) symphony swarm expediting

Creative (temp=2.0):
the according no waged over dug and venture 1949 man around he subsided '' eventually arose a sharply quickly shouting


# Convert Your Colab Notebook to PDF

### Step 1: Download Your Notebook
- Go to **File → Download → Download .ipynb**
- Save the file to your computer

### Step 2: Upload to Colab
- Click the **📁 folder icon** on the left sidebar
- Click the **upload button**
- Select your downloaded .ipynb file
- Wait for the upload to complete

### Step 3: Run the Code Below
- **Uncomment the cell below** and run the cell
- This will take about 1-2 minutes to install required packages

### Step 4: Enter Notebook Name
- When prompted, type your notebook name (e.g.`gs_000000_as2.ipynb`)
- Press Enter

### The PDF will automatically download to your computer

In [29]:
#  Install required packages (this takes about 30 seconds)
print("Installing PDF converter... please wait...")
!apt-get update -qq
!apt-get install -y texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc > /dev/null 2>&1
!pip install -q nbconvert

print("\n" + "="*50)
print("COLAB NOTEBOOK TO PDF CONVERTER")
print("="*50)
print("\nSTEP 1: Download your notebook")
print("- Go to File → Download → Download .ipynb")
print("- Save it to your computer")
print("\nSTEP 2: Upload it here")
print("- Click the folder icon on the left (📁)")
print("- Click the upload button and select your .ipynb file")
print("- Wait for upload to complete")
print("\nSTEP 3: Enter the filename below")
print("="*50)

#  Get notebook name from user
notebook_name = input("\nEnter your notebook name: ")

# Add .ipynb if missing
if not notebook_name.endswith('.ipynb'):
     notebook_name += '.ipynb'

import os
notebook_path = f'/content/{notebook_name}'

# Check if file exists
if not os.path.exists(notebook_path):
    print(f"\n⚠ Error: '{notebook_name}' not found in /content/")
    print("\nMake sure you uploaded the file using the folder icon (📁) on the left!")
else:
    print(f"\n✓ Found {notebook_name}")
    print("Converting to PDF... this may take 1-2 minutes...\n")

#  Convert the notebook to PDF
!jupyter nbconvert --to pdf "{notebook_path}"

#  Download the PDF
from google.colab import files
pdf_name = notebook_name.replace('.ipynb', '.pdf')
pdf_path = f'/content/{pdf_name}'

if os.path.exists(pdf_path):
   print("✓ SUCCESS! Downloading your PDF now...")
   files.download(pdf_path)
   print("\n✓ Done! Check your downloads folder.")
else:
   print("⚠ Error: Could not create PDF")

Installing PDF converter... please wait...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

COLAB NOTEBOOK TO PDF CONVERTER

STEP 1: Download your notebook
- Go to File → Download → Download .ipynb
- Save it to your computer

STEP 2: Upload it here
- Click the folder icon on the left (📁)
- Click the upload button and select your .ipynb file
- Wait for upload to complete

STEP 3: Enter the filename below

Enter your notebook name: AS3_2526622

⚠ Error: 'AS3_2526622.ipynb' not found in /content/

Make sure you uploaded the file using the folder icon (📁) on the left!
[NbConvertApp] WARNING | pattern '/content/AS3_2526622.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to